<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Space X  Falcon 9 First Stage Landing Prediction**


## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


Estimated time needed: **40** minutes


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


Falcon 9 first stage will land successfully


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


More specifically, the launch records are stored in a HTML table shown below:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


  ## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`: 
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame


First let's import required packages for this lab


In [1]:
!pip3 install beautifulsoup4
!pip3 install requests

In [2]:
import sys

import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

and we will provide some helper functions for you to process web scraped HTML table


In [3]:
def date_time(table_cells):
    """
    This function returns the data and time from the HTML  table cell
    Input: the  element of a table data cell extracts extra row
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """
    This function returns the booster version from the HTML  table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=''.join([booster_version for i,booster_version in enumerate( table_cells.strings) if i%2==0][0:-1])
    return out

def landing_status(table_cells):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=[i for i in table_cells.strings][0]
    return out


def get_mass(table_cells):
    mass=unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass=mass[0:mass.find("kg")+2]
    else:
        new_mass=0
    return new_mass


def extract_column_from_header(row):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
        
    colunm_name = ' '.join(row.contents)
    
    # Filter the digit and empty names
    if not(colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name    


To keep the lab tasks consistent, you will be asked to scrape the data from a snapshot of the  `List of Falcon 9 and Falcon Heavy launches` Wikipage updated on
`9th June 2021`


In [ ]:
static_url = "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; IBMDataScienceCapstone/1.0)"
}

Next, request the HTML page from the above URL and get a `response` object


### TASK 1: Request the Falcon9 Launch Wiki page from its URL


First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.


In [ ]:
# use requests.get() method with the provided static_url and headers
# assign the response to a object
response = requests.get(
    static_url,
    headers=headers,
    timeout=30
)
print(response.status_code)

Create a `BeautifulSoup` object from the HTML `response`


In [ ]:
# Use BeautifulSoup() to create a BeautifulSoup object from a response text content
soup = BeautifulSoup(response.content, "html.parser")

Print the page title to verify if the `BeautifulSoup` object was created properly 


In [4]:
# Use soup.title attribute
api_url = "https://en.wikipedia.org/w/api.php"

params = {
    "action": "parse",
    "page": "List_of_Falcon_9_and_Falcon_Heavy_launches",
    "prop": "text",
    "format": "json",
    "formatversion": "2"
}

headers = {
    "User-Agent": "IBMDataScienceCapstone/1.0"
}

response = requests.get(
    api_url,
    params=params,
    headers=headers,
    timeout=30
)

response.raise_for_status()

html = response.json()["parse"]["text"]
soup = BeautifulSoup(html, "html.parser")

print(soup.title)
print(len(soup.find_all("table")))

None
19


### TASK 2: Extract all column/variable names from the HTML table header


Next, we want to collect all relevant column names from the HTML table header


Let's try to find all tables on the wiki page first. If you need to refresh your memory about `BeautifulSoup`, please check the external reference link towards the end of this lab


In [20]:
# Use the find_all function in the BeautifulSoup object, with element type `table`
# Assign the result to a list called `html_tables`
html_tables = soup.find_all('table')

len(html_tables)

19

Starting from the third table is our target table contains the actual launch records.


In [21]:
# Let's print the third table and check its content
first_launch_table = html_tables[2]
print(first_launch_table)

<table class="wikitable plainrowheaders collapsible sticky-header" id="2026ytd" style="width: 100%;">
<tbody><tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date andtime ()
</th>
<th scope="col">
</th>
<th scope="col">Launchsite
</th>
<th scope="col">Payload
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launchoutcome
</th>
<th scope="col">
</th></tr>
<tr id="F9-583">
<th rowspan="2" scope="row" style="text-align:center;">583
</th>
<td><span data-sort-value="000000002026-01-03-0000" style="white-space:nowrap">January 3, 2026</span><br/>02:09<sup class="reference" id="cite_ref-264"><a href="#cite_note-264"><span class="cite-bracket">[</span>253<span class="cite-bracket">]</span></a></sup>
</td>
<td><a href="/wiki/Falcon_9_Block_5" title="Falcon 9 Block 5">F9<span class="nowrap"> </span>B5</a><br/><a href="/wiki/List_of_Falcon_9_first-stage_boosters#B1081" title="List of Falcon 9 first-stage boosters">B1081‑21</a>

You should able to see the columns names embedded in the table header elements `<th>` as follows:


```
<tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11">[b]</a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12">[c]</a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests">Booster<br/>landing</a>
</th></tr>
```


Next, we just need to iterate through the `<th>` elements and apply the provided `extract_column_from_header()` to extract column name one by one


In [22]:
column_names = [
    'Flight No.',
    'Date and time ( )',
    'Launch site',
    'Payload',
    'Payload mass',
    'Orbit',
    'Customer',
    'Launch outcome'
]

print(column_names)

['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


Check the extracted column names


In [15]:
print(column_names)

['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


## TASK 3: Create a data frame by parsing the launch HTML tables


We will create an empty dictionary with keys from the extracted column names in the previous task. Later, this dictionary will be converted into a Pandas dataframe


In [29]:
launch_dict = dict.fromkeys(column_names)

# Remove irrelevant combined date/time column
del launch_dict['Date and time ( )']

# Initialize columns as empty lists
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []

# Add new columns
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []

In [30]:
import re

extracted_row = 0

for table_number, table in enumerate(launch_tables):

    for rows in table.find_all("tr"):

        row = rows.find_all("td")

        # Skip rows that do not contain the required launch fields
        if not rows.th or len(row) < 9:
            continue

        # Get all visible text, including text inside nested HTML tags
        header_text = rows.th.get_text(" ", strip=True)

        # Extract a flight number from the beginning of the header
        match = re.match(r"^\s*(\d+)", header_text)

        if not match:
            continue

        flight_number = match.group(1)

        try:
            datatimelist = date_time(row[0])

            date = datatimelist[0].strip(",")
            time = datatimelist[1] if len(datatimelist) > 1 else ""

            bv = booster_version(row[1])
            if not bv:
                bv = row[1].a.get_text(strip=True) if row[1].a else ""

            launch_site = (
                row[2].a.get_text(strip=True)
                if row[2].a
                else row[2].get_text(" ", strip=True)
            )

            payload = (
                row[3].a.get_text(strip=True)
                if row[3].a
                else row[3].get_text(" ", strip=True)
            )

            payload_mass = get_mass(row[4])

            orbit = (
                row[5].a.get_text(strip=True)
                if row[5].a
                else row[5].get_text(" ", strip=True)
            )

            customer = (
                row[6].a.get_text(strip=True)
                if row[6].a
                else row[6].get_text(" ", strip=True)
            )

            launch_outcome = row[7].get_text(" ", strip=True)
            booster_landing = landing_status(row[8]).strip()

            # Append only after the complete row has parsed successfully
            launch_dict['Flight No.'].append(flight_number)
            launch_dict['Date'].append(date)
            launch_dict['Time'].append(time)
            launch_dict['Version Booster'].append(bv)
            launch_dict['Launch site'].append(launch_site)
            launch_dict['Payload'].append(payload)
            launch_dict['Payload mass'].append(payload_mass)
            launch_dict['Orbit'].append(orbit)
            launch_dict['Customer'].append(customer)
            launch_dict['Launch outcome'].append(launch_outcome)
            launch_dict['Booster landing'].append(booster_landing)

            extracted_row += 1

        except Exception as error:
            print(
                f"Skipped flight {flight_number}: "
                f"{type(error).__name__}: {error}"
            )

In [31]:
print("Rows extracted:", extracted_row)

for column, values in launch_dict.items():
    print(f"{column}: {len(values)}")

Rows extracted: 0
Flight No.: 0
Launch site: 0
Payload: 0
Payload mass: 0
Orbit: 0
Customer: 0
Launch outcome: 0
Version Booster: 0
Booster landing: 0
Date: 0
Time: 0


In [32]:
# Look at the first launch table and its first 10 rows
table = launch_tables[0]

for i, row in enumerate(table.find_all("tr")[:10]):
    print("\nROW", i)
    print("TH:", repr(row.th.get_text(" ", strip=True)) if row.th else None)
    print("TD COUNT:", len(row.find_all("td")))

IndexError: list index out of range

In [33]:
print(response.status_code)
print(response.url)
print(soup.title)

200
https://en.wikipedia.org/w/api.php?action=parse&page=List_of_Falcon_9_and_Falcon_Heavy_launches&prop=text&format=json&formatversion=2
None


In [34]:
all_tables = soup.find_all('table')

print("Total tables:", len(all_tables))

Total tables: 19


In [35]:
for i, table in enumerate(all_tables):
    print(i, table.get('class'))


0 ['toc']
1 ['wikitable', 'plainrowheaders', 'collapsible', 'sticky-header']
2 ['wikitable', 'plainrowheaders', 'collapsible', 'sticky-header']
3 ['wikitable', 'sticky-header']
4 ['wikitable', 'sticky-header']
5 ['nowraplinks', 'hlist', 'mw-collapsible', 'autocollapse', 'navbox-inner']
6 ['nowraplinks', 'navbox-subgroup']
7 ['nowraplinks', 'navbox-subgroup']
8 ['nowraplinks', 'navbox-subgroup']
9 ['nowraplinks', 'mw-collapsible', 'mw-collapsed', 'navbox-inner']
10 ['nowraplinks', 'navbox-subgroup']
11 ['nowraplinks', 'navbox-subgroup']
12 ['nowraplinks', 'navbox-subgroup']
13 ['nowraplinks', 'hlist', 'mw-collapsible', 'autocollapse', 'navbox-inner']
14 ['nowraplinks', 'navbox-subgroup']
15 ['nowraplinks', 'navbox-subgroup']
16 ['nowraplinks', 'navbox-subgroup']
17 ['nowraplinks', 'navbox-subgroup']
18 ['nowraplinks', 'navbox-subgroup']


In [36]:
first_launch_table = all_tables[2]

print(first_launch_table.get('class'))

['wikitable', 'plainrowheaders', 'collapsible', 'sticky-header']


In [37]:
rows = first_launch_table.find_all('tr')

print("Rows:", len(rows))

Rows: 244


In [38]:
for i, row in enumerate(rows[:5]):
    print("\nROW", i)
    print("TH:", row.find('th').get_text(" ", strip=True)
          if row.find('th') else None)
    print("TD COUNT:", len(row.find_all('td')))


ROW 0
TH: Flight No.
TD COUNT: 0

ROW 1
TH: 583
TD COUNT: 9

ROW 2
TH: None
TD COUNT: 1

ROW 3
TH: 584
TD COUNT: 9

ROW 4
TH: None
TD COUNT: 1


In [39]:
import requests
from bs4 import BeautifulSoup

static_url = (
    "https://en.wikipedia.org/w/index.php?"
    "title=List_of_Falcon_9_and_Falcon_Heavy_launches"
    "&oldid=1027686922"
)

headers = {
    "User-Agent": "IBMDataScienceCapstone/1.0"
}

response = requests.get(
    static_url,
    headers=headers,
    timeout=30
)

response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

print("Status:", response.status_code)
print("Final URL:", response.url)
print("Title:", soup.title)


Status: 200
Final URL: https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922
Title: <title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>


In [40]:
html_tables = soup.find_all("table")

print("Total tables:", len(html_tables))

first_launch_table = html_tables[2]
rows = first_launch_table.find_all("tr")

for i, row in enumerate(rows[:5]):
    th = row.find("th")
    
    print(
        "ROW", i,
        "| TH:", th.get_text(" ", strip=True) if th else None,
        "| TD COUNT:", len(row.find_all("td"))
    )

Total tables: 29
ROW 0 | TH: Flight No. | TD COUNT: 0
ROW 1 | TH: 1 | TD COUNT: 9
ROW 2 | TH: None | TD COUNT: 1
ROW 3 | TH: 2 | TD COUNT: 9
ROW 4 | TH: None | TD COUNT: 1


In [41]:
launch_tables = soup.find_all(
    "table",
    class_="wikitable plainrowheaders collapsible"
)

print("Launch tables:", len(launch_tables))

Launch tables: 9


In [42]:
count = 0

for table in launch_tables:
    for row in table.find_all("tr"):
        th = row.find("th")
        cells = row.find_all("td")

        if th:
            flight_number = th.get_text(strip=True)

            if flight_number.isdigit() and len(cells) >= 9:
                count += 1

print("Launch rows found:", count)

Launch rows found: 121


In [48]:
launch_dict = {
    "Flight No.": [],
    "Launch site": [],
    "Payload": [],
    "Payload mass": [],
    "Orbit": [],
    "Customer": [],
    "Launch outcome": [],
    "Version Booster": [],
    "Booster landing": [],
    "Date": [],
    "Time": []
}

Next, we just need to fill up the `launch_dict` with launch records extracted from table rows.


Usually, HTML tables in Wiki pages are likely to contain unexpected annotations and other types of noises, such as reference links `B0004.1[8]`, missing values `N/A [e]`, inconsistent formatting, etc.


To simplify the parsing process, we have provided an incomplete code snippet below to help you to fill up the `launch_dict`. Please complete the following code snippet with TODOs or you can choose to write your own logic to parse all launch tables:


In [44]:
tables_to_parse = soup.find_all(
    'table',
    "wikitable plainrowheaders collapsible"
)

print("Tables found:", len(tables_to_parse))

Tables found: 9


In [50]:
extracted_row = 0
#Extract each table 
for table_number,table in enumerate(soup.find_all('table',"wikitable plainrowheaders collapsible")):
   # get table row 
    for rows in table.find_all("tr"):
        #check to see if first table heading is as number corresponding to launch a number 
        if rows.th:
            if rows.th.string:
                flight_number=rows.th.string.strip()
                flag=flight_number.isdigit()
        else:
            flag=False
        #get table element 
        row=rows.find_all('td')
        #if it is number save cells in a dictonary 
        if flag:
            extracted_row += 1
            # Flight Number value
            # TODO: Append the flight_number into launch_dict with key `Flight No.`
            #print(flight_number)
            datatimelist=date_time(row[0])
            
            # Date value
            # TODO: Append the date into launch_dict with key `Date`
            date = datatimelist[0].strip(',')
            #print(date)
            
            # Time value
            # TODO: Append the time into launch_dict with key `Time`
            time = datatimelist[1]
            #print(time)
              
            # Booster version
            # TODO: Append the bv into launch_dict with key `Version Booster`
            bv=booster_version(row[1])
            if not(bv):
                bv=row[1].a.string
            print(bv)
            
            # Launch Site
            # TODO: Append the bv into launch_dict with key `Launch Site`
            launch_site = row[2].a.string
            #print(launch_site)
            
            # Payload
            # TODO: Append the payload into launch_dict with key `Payload`
            payload = row[3].a.string
            #print(payload)
            
            # Payload Mass
            # TODO: Append the payload_mass into launch_dict with key `Payload mass`
            payload_mass = get_mass(row[4])
            #print(payload)
            
            # Orbit
            # TODO: Append the orbit into launch_dict with key `Orbit`
            orbit = row[5].a.string
            #print(orbit)
            
            # Customer
            # TODO: Append the customer into launch_dict with key `Customer`
            customer = (
    row[6].a.get_text(strip=True)
    if row[6].a
    else row[6].get_text(" ", strip=True)
)
            #print(customer)
            
            # Launch outcome
            # TODO: Append the launch_outcome into launch_dict with key `Launch outcome`
            launch_outcome = list(row[7].strings)[0]
            #print(launch_outcome)
            
            # Booster landing
            # TODO: Append the launch_outcome into launch_dict with key `Booster landing`
            booster_landing = landing_status(row[8])
            #print(booster_landing)
            

F9 v1.07B0003.1
F9 v1.07B0004.1
F9 v1.07B0005.1
F9 v1.07B0006.1
F9 v1.07B0007.1
F9 v1.17B1003
F9 v1.1
F9 v1.1
F9 v1.1
F9 v1.1
F9 v1.1
F9 v1.1[
F9 v1.1[
F9 v1.1[
F9 v1.1[
F9 v1.1[
F9 v1.1[
F9 v1.1[
F9 v1.1[
F9 FT[
F9 v1.1[
F9 FT[
F9 FT[
F9 FT[
F9 FT[
F9 FT[
F9 FT[
F9 FT[
F9 FT[
F9 FT[
F9 FT[
F9 FT♺[
F9 FT[
F9 FT[
F9 FT[
F9 FTB1029.2
F9 FT[
F9 FT[
F9 B4[
F9 FT[
F9 B4[
F9 B4[
F9 FTB1031.2
F9 B4[
F9 FTB1035.2
F9 FTB1036.2
F9 B4[
F9 FTB1032.2
F9 FTB1038.2
F9 B4[
F9 B4B1041.2
F9 B4B1039.2
F9 B4[
F9 B5311B1046.1
F9 B4B1043.2
F9 B4B1040.2
F9 B4B1045.2
F9 B5
F9 B5349B1048[
F9 B5B1046.2
F9 B5[
F9 B5B1048.2
F9 B5B1047.2
F9 B5B1046.3268
F9 B5[
F9 B5[
F9 B5B1049.2
F9 B5B1048.3
F9 B5[]
F9 B5[
F9 B5B1049.3
F9 B5B1051.2
F9 B5B1056.2
F9 B5B1047.3
F9 B5
F9 B5[
F9 B5B1056.3
F9 B5
F9 B5
F9 B5
F9 B5
F9 B5
F9 B5
F9 B5
F9 B5[
F9 B5
F9 B5
F9 B5
F9 B5B1058.2
F9 B5
F9 B5B1049.6
F9 B5
F9 B5B1060.2
F9 B5B1058.3
F9 B5B1051.6
F9 B5
F9 B5
F9 B5[
F9 B5
F9 B5 ♺[
F9 B5 ♺[
F9 B5 ♺
F9 B5 ♺
F9 B5
F9 B5B1051.8
F9 B5B1058.5

In [53]:
print("Rows extracted:", extracted_row)

for column, values in launch_dict.items():
    print(column, len(values))

Rows extracted: 121
Flight No. 0
Launch site 0
Payload 0
Payload mass 0
Orbit 0
Customer 0
Launch outcome 0
Version Booster 0
Booster landing 0
Date 0
Time 0


In [54]:
launch_dict = {
    'Flight No.': [],
    'Launch site': [],
    'Payload': [],
    'Payload mass': [],
    'Orbit': [],
    'Customer': [],
    'Launch outcome': [],
    'Version Booster': [],
    'Booster landing': [],
    'Date': [],
    'Time': []
}

extracted_row = 0

In [55]:
for table in launch_tables:
    for rows in table.find_all("tr"):

        if rows.th:
            flight_number = rows.th.get_text(strip=True)

            if flight_number.isdigit():
                extracted_row += 1
                launch_dict['Flight No.'].append(flight_number)

print("Rows identified:", extracted_row)
print("Flight numbers appended:", len(launch_dict['Flight No.']))
print(launch_dict['Flight No.'][:10])

Rows identified: 121
Flight numbers appended: 121
['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']


In [56]:
launch_dict = {
    'Flight No.': [],
    'Launch site': [],
    'Payload': [],
    'Payload mass': [],
    'Orbit': [],
    'Customer': [],
    'Launch outcome': [],
    'Version Booster': [],
    'Booster landing': [],
    'Date': [],
    'Time': []
}

extracted_row = 0

In [57]:
for table in launch_tables:
    for rows in table.find_all("tr"):

        row = rows.find_all("td")

        if not rows.th:
            continue

        flight_number = rows.th.get_text(strip=True)

        # Only process actual launch rows
        if not flight_number.isdigit() or len(row) < 9:
            continue

        # Extract values first
        datatimelist = date_time(row[0])
        date = datatimelist[0].strip(',')
        time = datatimelist[1]

        bv = booster_version(row[1])
        if not bv:
            bv = row[1].get_text(" ", strip=True)

        launch_site = (
            row[2].a.get_text(strip=True)
            if row[2].a
            else row[2].get_text(" ", strip=True)
        )

        payload = (
            row[3].a.get_text(strip=True)
            if row[3].a
            else row[3].get_text(" ", strip=True)
        )

        payload_mass = get_mass(row[4])

        orbit = (
            row[5].a.get_text(strip=True)
            if row[5].a
            else row[5].get_text(" ", strip=True)
        )

        customer = (
            row[6].a.get_text(strip=True)
            if row[6].a
            else row[6].get_text(" ", strip=True)
        )

        launch_outcome = row[7].get_text(" ", strip=True)

        booster_landing = landing_status(row[8]).strip()

        # Append only after everything above succeeded
        launch_dict['Flight No.'].append(flight_number)
        launch_dict['Date'].append(date)
        launch_dict['Time'].append(time)
        launch_dict['Version Booster'].append(bv)
        launch_dict['Launch site'].append(launch_site)
        launch_dict['Payload'].append(payload)
        launch_dict['Payload mass'].append(payload_mass)
        launch_dict['Orbit'].append(orbit)
        launch_dict['Customer'].append(customer)
        launch_dict['Launch outcome'].append(launch_outcome)
        launch_dict['Booster landing'].append(booster_landing)

        extracted_row += 1

In [58]:
print("Rows extracted:", extracted_row)

for column, values in launch_dict.items():
    print(column, len(values))

Rows extracted: 121
Flight No. 121
Launch site 121
Payload 121
Payload mass 121
Orbit 121
Customer 121
Launch outcome 121
Version Booster 121
Booster landing 121
Date 121
Time 121


In [59]:
df = pd.DataFrame(launch_dict)

print(df.shape)
display(df.head())

(121, 11)


,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,F9 v1.07B0003.1,Failure,4 June 2010,18:45
1,2,CCAFS,Dragon,0,LEO,NASA,Success [ 9 ],F9 v1.07B0004.1,Failure,8 December 2010,15:43
2,3,CCAFS,Dragon,525 kg,LEO,NASA,Success [ 20 ],F9 v1.07B0005.1,No,22 May 2012,07:44
3,4,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success,F9 v1.07B0006.1,No attempt,8 October 2012,00:35
4,5,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success,F9 v1.07B0007.1,No,1 March 2013,15:10


After you have fill in the parsed launch record values into `launch_dict`, you can create a dataframe from it.


In [51]:
df= pd.DataFrame({ key:pd.Series(value) for key, value in launch_dict.items() })

We can now export it to a <b>CSV</b> for the next section, but to make the answers consistent and in case you have difficulties finishing this lab. 

Following labs will be using a provided dataset to make each lab independent. 


<code>df.to_csv('spacex_web_scraped.csv', index=False)</code>


In [60]:
df.to_csv('spacex_web_scraped.csv', index=False)

## Authors


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


<!--
## Change Log
-->


<!--
| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-06-09        | 1.0     | Yan Luo    | Tasks updates           |
| 2020-11-10        | 1.0     | Nayef      | Created the initial version |
-->


Copyright © 2021 IBM Corporation. All rights reserved.
